# 言葉で計算する — Word2Vec を好きなだけ動かす

図解に積めたのは 15,000 語まででした。ここでは **40 万語** をまるごと読み込みます。
思いついた語を、片端から入れて試せます。

英語（GloVe）と日本語（chiVe）を切り替えられます。
同じ計算が言語によって崩れるところまで見ます。

- **戻る**: [言葉を、場所に変える](https://manga-epoch.github.io/viewer/pub/epoch/arc2/figures_embed.html) — 同じ内容をスライダで動かせます
- **必要なもの**: Colab の標準環境（ダウンロードに数分・GPU 不要）
- 上から順に実行してください（Colab では `Shift + Enter`）。

## 0. 準備 — 40 万語を読み込む

まず英語から。GloVe（Wikipedia + Gigaword、100 次元、40 万語）を落とします。
128MB あるので 1〜2 分かかります。

配布データは **PDDL 1.0（パブリックドメイン）** で、自由に使えます。

In [ ]:
%pip install -q gensim

In [ ]:
import numpy as np, urllib.request, gzip, os

URL_EN = ("https://github.com/RaRe-Technologies/gensim-data/releases/download/"
          "glove-wiki-gigaword-100/glove-wiki-gigaword-100.gz")

def load_glove(path="glove100.gz"):
    if not os.path.exists(path):
        print("ダウンロード中…（1〜2 分）")
        urllib.request.urlretrieve(URL_EN, path)
    words, vecs = [], []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        head = f.readline().split()
        rows = [] if len(head) == 2 else [" ".join(head)]
        for line in rows + list(f):
            p = line.rstrip().split(" ")
            words.append(p[0]); vecs.append(np.asarray(p[1:], dtype=np.float32))
    V = np.vstack(vecs)
    return words, V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)

WORDS, V = load_glove()
IDX = {w: i for i, w in enumerate(WORDS)}
print(f"{len(WORDS):,} 語 × {V.shape[1]} 次元")

## 1. 近さを測る

意味の近さは **コサイン類似度**――2 本のベクトルがなす角度で測ります。
長さを 1 に揃えてあるので、内積がそのまま類似度になります。

$$\mathrm{sim}(a,b)=\frac{a\cdot b}{\lVert a\rVert\,\lVert b\rVert}$$

In [ ]:
def near(word, topn=10):
    if word not in IDX:
        return f"{word!r} は語彙にありません"
    s = V @ V[IDX[word]]
    s[IDX[word]] = -9
    return [(WORDS[i], round(float(s[i]), 3)) for i in np.argsort(-s)[:topn]]

for w in ["king", "tokyo", "guitar", "coffee", "python"]:
    print(f"{w:9}: {[x for x, _ in near(w, 8)]}")

### 検算 — この類似度は正しいか

自分で書いた内積が、ライブラリ（gensim）の答えと一致するか確かめます。
一致しなければ、正規化かインデックスのどこかが間違っています。

In [ ]:
from gensim.models import KeyedVectors

kv = KeyedVectors(vector_size=V.shape[1])
kv.add_vectors(WORDS, V)
mine = [w for w, _ in near("king", 5)]
theirs = [w for w, _ in kv.most_similar("king", topn=5)]
print("自前  :", mine)
print("gensim:", theirs)
print("→ 一致" if mine == theirs else "→ 不一致。正規化かインデックスを見直す")

## 2. 意味を足し引きする

**ここが Word2Vec のいちばん有名なところです。** 好きな語で試してください。

$$v = \mathrm{vec}(B) - \mathrm{vec}(A) + \mathrm{vec}(C)$$

In [ ]:
def analogy(b, a, c, topn=5, words=None, mat=None, idx=None):
    words = words or WORDS; mat = V if mat is None else mat; idx = idx or IDX
    if any(w not in idx for w in (a, b, c)):
        return "語彙にない語: " + ", ".join(w for w in (a, b, c) if w not in idx)
    v = mat[idx[b]] - mat[idx[a]] + mat[idx[c]]
    v = v / np.linalg.norm(v)
    s = mat @ v
    for w in (a, b, c): s[idx[w]] = -9
    return [(words[i], round(float(s[i]), 3)) for i in np.argsort(-s)[:topn]]

for b, a, c in [("king","man","woman"), ("paris","france","japan"),
                ("actor","man","woman"), ("better","good","bad"),
                ("windows","microsoft","apple"), ("sushi","japan","italy")]:
    print(f"{b} - {a} + {c:9} = {[w for w, _ in analogy(b, a, c, 3)]}")

In [ ]:
# ← ここを書き換えて、好きな語で試してください
analogy("einstein", "science", "music", topn=8)

### どれくらい当たるのか

1 例で「動いた」と言うのは危ういので、まとめて解かせて正解率を出します。

In [ ]:
QUIZ = [("king","man","woman","queen"), ("actor","man","woman","actress"),
        ("tokyo","japan","france","paris"), ("rome","italy","spain","madrid"),
        ("better","good","bad","worse"), ("cats","cat","dog","dogs"),
        ("smaller","small","big","bigger"), ("french","france","japan","japanese"),
        ("sister","brother","son","daughter"), ("walked","walking","swimming","swam"),
        ("berlin","germany","russia","moscow"), ("uncle","man","woman","aunt"),
        ("shortest","short","tall","tallest"), ("mice","mouse","goose","geese"),
        ("nephew","boy","girl","niece"), ("dollars","dollar","euro","euros"),
        ("copenhagen","denmark","norway","oslo"), ("cheaper","cheap","expensive","costlier"),
        ("daughters","daughter","son","sons"), ("thinking","think","read","reading")]

hit = []
for b, a, c, want in QUIZ:
    r = analogy(b, a, c, 1)
    got = r[0][0] if isinstance(r, list) else None
    hit.append(got == want)
    if got != want:
        print(f"  ✗ {b} - {a} + {c} = {got}（想定 {want}）")
print(f"\n正解 {sum(hit)}/{len(QUIZ)} = {sum(hit)/len(QUIZ)*100:.0f}%")
print("外した問題も、意味としては近いものが多い。「必ず当たる魔法」ではありません。")

## 3. 地図にする

100 次元は目で見られないので、**PCA** で 2 次元に落として並べます。
落とすときに情報は失われますが、まとまり方の傾向は見えます。

In [ ]:
import matplotlib.pyplot as plt

GROUPS = {
    "country": ["japan","france","germany","italy","spain","china","brazil","canada"],
    "capital": ["tokyo","paris","berlin","rome","madrid","beijing","brasilia","ottawa"],
    "animal":  ["dog","cat","horse","tiger","elephant","rabbit","wolf","bear"],
    "food":    ["bread","rice","cheese","apple","soup","cake","coffee","wine"],
}
sel = [w for g in GROUPS.values() for w in g if w in IDX]
X = V[[IDX[w] for w in sel]]
X = X - X.mean(0)
P = X @ np.linalg.svd(X, full_matrices=False)[2][:2].T

plt.figure(figsize=(9, 7))
for (name, g), c in zip(GROUPS.items(), ["#6E7BA8", "#B4493F", "#5b8266", "#a08040"]):
    pts = np.array([P[sel.index(w)] for w in g if w in sel])
    plt.scatter(pts[:, 0], pts[:, 1], s=60, c=c, label=name)
for w, (x, y) in zip(sel, P):
    plt.annotate(w, (x, y), fontsize=8, xytext=(4, 3), textcoords="offset points")
# 国と首都を線で結ぶ。平行に並べば「首都である」という関係が向きになっている証拠
for a, b in zip(GROUPS["country"], GROUPS["capital"]):
    if a in sel and b in sel:
        pa, pb = P[sel.index(a)], P[sel.index(b)]
        plt.plot([pa[0], pb[0]], [pa[1], pb[1]], color="#cfccc4", lw=1, zorder=0)
plt.legend(); plt.title("PCA of GloVe vectors (country - capital pairs linked)")
plt.tight_layout(); plt.show()

国と首都を結ぶ線が、だいたい**同じ向き・同じ長さ**に並びます。
「首都である」という関係が、空間の中の 1 本の向きとして表れている――
これが足し引きできる理由です。

## 4. 日本語でやる

日本語には **chiVe**（国立国語研究所・ワークスアプリケーションズ / Apache License 2.0）を
使います。41 万語・300 次元。ダウンロードが 0.5GB あるので、数分かかります。

In [ ]:
import tarfile

URL_JA = ("https://sudachi.s3-ap-northeast-1.amazonaws.com/chive/"
          "chive-1.3-mc90_gensim.tar.gz")

if not os.path.exists("chive-1.3-mc90_gensim"):
    print("ダウンロード中…（0.5GB・数分）")
    urllib.request.urlretrieve(URL_JA, "chive.tar.gz")
    tarfile.open("chive.tar.gz").extractall(".")

kv_ja = KeyedVectors.load("chive-1.3-mc90_gensim/chive-1.3-mc90.kv")
WORDS_JA = list(kv_ja.index_to_key)
V_JA = np.asarray(kv_ja.vectors, dtype=np.float32)
V_JA = V_JA / (np.linalg.norm(V_JA, axis=1, keepdims=True) + 1e-9)
IDX_JA = {w: i for i, w in enumerate(WORDS_JA)}
print(f"{len(WORDS_JA):,} 語 × {V_JA.shape[1]} 次元")

In [ ]:
def near_ja(w, topn=8):
    if w not in IDX_JA: return f"{w!r} は語彙にありません"
    s = V_JA @ V_JA[IDX_JA[w]]; s[IDX_JA[w]] = -9
    return [WORDS_JA[i] for i in np.argsort(-s)[:topn]]

def ana_ja(b, a, c, topn=5):
    return analogy(b, a, c, topn, WORDS_JA, V_JA, IDX_JA)

for w in ["猫", "音楽", "東京", "人工知能"]:
    print(f"{w:6}: {near_ja(w)}")
print()
for b, a, c in [("俳優","男性","女性"), ("姉","兄","弟"), ("東京","日本","フランス")]:
    print(f"{b} - {a} + {c} = {[w for w, _ in ana_ja(b, a, c, 3)]}")

### 同じ計算が、言語を変えると崩れる

英語で有名な `king - man + woman = queen` を、日本語でそのままやってみます。

In [ ]:
print("英語 :", [w for w, _ in analogy("king", "man", "woman", 3)])
print("日本語:", [w for w, _ in ana_ja("王", "男", "女", 3)])
print("\n日本語の「王」に近い語:", near_ja("王"))

日本語では **女王 が出ません**。理由は「王」の近くに並ぶ語を見ると分かります――
「王様」「帝」「魔王」「大王」と、**称号や物語の語**ばかりです。
「女王」と対になる「男の王」という使われ方が薄いので、男女の差を引き算で取り出せない。

日本語が悪いのでも Word2Vec が悪いのでもありません。
**学習に使った文章に、その語がどう出てきたかが、そのまま出ている**だけです。
これは次の節につながります。

なお図解の方は 15,000 語に絞った版を積んでいるので、同じ計算でも答えが変わります
（そちらでは「帝」が出ます）。**語彙をどこで切るかでも結果は動く**、ということです。

## 5. 鏡には、偏りも写る

単語ベクトルは集めた文章の鏡なので、そこにある偏りもそのまま学習します。
「そういうこともある」で済ませず、**測ります**。

In [ ]:
for b, a, c in [("doctor","man","woman"), ("engineer","man","woman"),
                ("nurse","woman","man"), ("boss","man","woman")]:
    print(f"{b} - {a} + {c:6} = {[w for w, _ in analogy(b, a, c, 3)]}")

In [ ]:
# 職業名が「男性側/女性側」どちらに寄っているかを、1 本の軸で測る
axis = V[IDX["woman"]] - V[IDX["man"]]
axis /= np.linalg.norm(axis)
jobs = ["nurse","teacher","librarian","secretary","dancer","doctor","lawyer",
        "scientist","engineer","programmer","carpenter","pilot","surgeon","plumber"]
sc = sorted(((float(V[IDX[j]] @ axis), j) for j in jobs if j in IDX), reverse=True)
for s, j in sc:
    bar = "█" * int(abs(s) * 220)
    print(f"  {j:12} {s:+.3f} {'女性側 ' + bar if s > 0 else '男性側 ' + bar}")

**逆に見える結果**も混じります。たとえば `secretary`（秘書）が男性側に出ます。
近い語を見ると `deputy, general, minister, chief, vice` ――
このデータ（Wikipedia + ニュース）では「秘書」ではなく **長官・次官**の意味が優勢だからです。

これも同じ話です。ベクトルは「社会一般」を映しているのではなく、
**このデータに書かれていたこと**を映しています。だから別のデータで学習すれば別の軸が出ます。

In [ ]:
print("secretary に近い語:", [w for w, _ in near("secretary", 8)])

値そのものより、**この軸が引けてしまうこと**が問題です。
求人の並べ替えや履歴書の選別にこのベクトルを使えば、偏りはそのまま結果に出ます。

対処は「偏りのない完璧なデータを集める」ことではありません（それは無理です）。
**偏りがあることを測って把握し、用途ごとに影響を評価する**のが実務の作法です。

## 次に

- **図解に戻る**: [言葉を、場所に変える](https://manga-epoch.github.io/viewer/pub/epoch/arc2/figures_embed.html)
- **文脈で意味が変わる仕組みへ**: [attention.ipynb](https://colab.research.google.com/github/manga-epoch/viewer/blob/main/notebooks/attention.ipynb)
  ―― 同じ「はし」でも前後の語によって表現が変わる、その先の話

---

## 出典とライセンス

このノートブックは、マンガ **EPOCH — 時代の前夜** の④「書く」レイヤーです。
[EPOCH について](https://manga-epoch.github.io/viewer)

英語ベクトルは **GloVe**（Stanford NLP、配布データは Public Domain Dedication and License v1.0）、日本語ベクトルは **chiVe**（国立国語研究所・株式会社ワークスアプリケーションズ、Apache License 2.0）。

本ノートブックのコードは自由に改変して使えます。